# 第8课：Fixed 50% Hedge与Futures P&L

本课第一次真正加入套保。策略规则很简单：在3月按预计产量的50%卖出December corn futures，并一直持有到收获期。

本课会严格拆开教授要求的两个损益来源：

1. Initial futures position P&L；
2. July adjustment P&L。

对于固定策略，第2项为0，但仍然必须在模型中单独显示。

## 0. 套保时间线

### 3月

- 预计2026产量；
- 卖出相当于预计产量50%的December futures；
- 成交价格为每个情景相同的 $F_0=\$4.70/bu$。

### 7月

- 可以看到July天气与价格；
- 但Fixed 50%策略不改变合约数量，所以adjustment = 0。

### 收获期

- 按cash price出售实际玉米；
- 按Harvest futures price平掉期货空头；
- 合并cash revenue、futures P&L和实施成本。

## 1. 为什么卖出期货可以保护价格下跌？

农民未来会卖出玉米，因此担心价格下跌。套保时先**卖出/做空**期货：

$$InitialFuturesPnL_i=N_0\times Q\times(F_0-F_{H,i})$$

- $N_0$：3月卖出的合约数；
- $Q$：每份合约5,000 bushels；
- $F_0$：3月卖出价格；
- $F_H$：收获期买回平仓价格。

如果收获期期货价格下降，$F_0-F_H>0$，期货头寸盈利；如果价格上涨，期货头寸亏损。

## 2. 导入工具并锁定实施假设

In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

N_SIMULATIONS = 10_000
FARM_ACRES = 1_000
HEDGE_RATIO = 0.50
CONTRACT_SIZE_BUSHELS = 5_000

TRANSACTION_COST_PER_CONTRACT_SIDE = 25.00
INITIAL_MARGIN_PER_CONTRACT = 2_500.00
MARGIN_FINANCING_RATE_ANNUAL = 0.06
DAYS_PRESEASON_TO_JULY = 136
DAYS_JULY_TO_HARVEST = 108
MARGIN_LIQUIDITY_RESERVE = 50_000.00

LOWER_TAIL_PROBABILITY = 0.05

print('策略: Fixed 50%')
print('合约规模:', CONTRACT_SIZE_BUSHELS, 'bushels')
print('Transaction cost: $', TRANSACTION_COST_PER_CONTRACT_SIDE, 'per contract-side')
print('Initial margin: $', INITIAL_MARGIN_PER_CONTRACT, 'per contract')

## 3. 读取第7课结果

请先运行第7课，使当前工作目录中存在：

`lesson_07_outputs/unhedged_baseline_10000.csv`

In [ ]:
candidate_paths = [
    Path.cwd() / 'lesson_07_outputs' / 'unhedged_baseline_10000.csv',
    Path.cwd().parent / 'lesson_07_outputs' / 'unhedged_baseline_10000.csv',
]

lesson7_path = next((p for p in candidate_paths if p.exists()), None)

if lesson7_path is None:
    raise FileNotFoundError(
        '没有找到第7课CSV。请先运行Lesson_07 Notebook的全部单元格，'
        '并保留lesson_07_outputs文件夹。'
    )

scenarios = pd.read_csv(lesson7_path)
print('读取文件:', lesson7_path)
print('行数:', len(scenarios))
print('列数:', len(scenarios.columns))

## 4. 检查本课输入

In [ ]:
required_columns = [
    'scenario_id',
    'preseason_expected_yield_bu_per_acre',
    'final_yield_bu_per_acre',
    'actual_production_bushels',
    'preseason_futures_usd_per_bushel',
    'july_futures_usd_per_bushel',
    'harvest_futures_usd_per_bushel',
    'cash_revenue_usd',
    'production_cost_usd',
    'unhedged_profit_usd',
    'unhedged_profit_usd_per_acre',
]

missing = [c for c in required_columns if c not in scenarios.columns]
assert not missing, f'缺少列: {missing}'
assert len(scenarios) == N_SIMULATIONS
assert scenarios['scenario_id'].is_unique
assert scenarios[required_columns].isna().sum().sum() == 0

print('第7课输入检查通过。')

## 5. 计算3月需要卖出多少份合约

$$ExpectedProduction=ExpectedYield\times Acres$$

$$TargetHedgedBushels=50\%\times ExpectedProduction$$

$$InitialContracts=Round\left(\frac{TargetHedgedBushels}{5{,}000}\right)$$

代码使用`floor(x + 0.5)`实现非负数量的nearest-whole-contract rounding，避免Python内置`round`的banker's rounding。

In [ ]:
expected_yield = scenarios['preseason_expected_yield_bu_per_acre'].to_numpy()
expected_production = expected_yield * FARM_ACRES
target_hedged_bushels = HEDGE_RATIO * expected_production

initial_contracts = np.floor(
    target_hedged_bushels / CONTRACT_SIZE_BUSHELS + 0.5
).astype(int)

scenarios['fixed50_initial_hedge_ratio'] = HEDGE_RATIO
scenarios['fixed50_initial_contracts'] = initial_contracts
scenarios['fixed50_initial_hedged_bushels'] = (
    initial_contracts * CONTRACT_SIZE_BUSHELS
)

print(f'预计产量 = {expected_production[0]:,.3f} bushels')
print(f'50%目标套保量 = {target_hedged_bushels[0]:,.3f} bushels')
print('四舍五入后的合约数 =', initial_contracts[0])
print('实际套保bushels =', scenarios.loc[0, 'fixed50_initial_hedged_bushels'])
print(f"离散合约后的实际初始比例 = {scenarios.loc[0, 'fixed50_initial_hedged_bushels'] / expected_production[0]:.4%}")

## 6. 明确记录July Adjustment

统一的两段P&L公式是：

$$InitialPnL_i=N_{0,i}\times Q\times(F_{0,i}-F_{H,i})$$

$$JulyAdjustmentPnL_i=\Delta N_{July,i}\times Q\times(F_{July,i}-F_{H,i})$$

Fixed 50%在7月不调整，所以：

$$\Delta N_{July,i}=0$$

$$JulyAdjustmentPnL_i=0$$

把0明确写出来，可以证明模型没有把3月和7月交易价格混在一起。

In [ ]:
july_adjustment_contracts = np.zeros(N_SIMULATIONS, dtype=int)
final_contracts = initial_contracts + july_adjustment_contracts

scenarios['fixed50_july_adjustment_contracts'] = july_adjustment_contracts
scenarios['fixed50_final_contracts'] = final_contracts
scenarios['fixed50_final_hedged_bushels'] = final_contracts * CONTRACT_SIZE_BUSHELS

print('Initial contracts unique values:', np.unique(initial_contracts))
print('July adjustment unique values:', np.unique(july_adjustment_contracts))
print('Final contracts unique values:', np.unique(final_contracts))

## 7. 计算教授要求的两段Futures P&L

In [ ]:
f0 = scenarios['preseason_futures_usd_per_bushel'].to_numpy()
f_july = scenarios['july_futures_usd_per_bushel'].to_numpy()
f_harvest = scenarios['harvest_futures_usd_per_bushel'].to_numpy()

initial_futures_pnl = (
    initial_contracts
    * CONTRACT_SIZE_BUSHELS
    * (f0 - f_harvest)
)

july_adjustment_pnl = (
    july_adjustment_contracts
    * CONTRACT_SIZE_BUSHELS
    * (f_july - f_harvest)
)

total_futures_pnl = initial_futures_pnl + july_adjustment_pnl

scenarios['fixed50_initial_futures_pnl_usd'] = initial_futures_pnl
scenarios['fixed50_july_adjustment_pnl_usd'] = july_adjustment_pnl
scenarios['fixed50_total_futures_pnl_usd'] = total_futures_pnl

print(scenarios[[
    'scenario_id',
    'preseason_futures_usd_per_bushel',
    'july_futures_usd_per_bushel',
    'harvest_futures_usd_per_bushel',
    'fixed50_initial_futures_pnl_usd',
    'fixed50_july_adjustment_pnl_usd',
    'fixed50_total_futures_pnl_usd',
]].head(10).round(2).to_string(index=False))

## 8. 用Scenario 1手算Futures P&L

Scenario 1中，3月以$4.70卖出21份合约，收获期以约$3.99194买回，因此：

$$21\times5{,}000\times(4.70-3.99194)$$

In [ ]:
row1 = scenarios.iloc[0]
manual_initial_pnl = (
    row1['fixed50_initial_contracts']
    * CONTRACT_SIZE_BUSHELS
    * (
        row1['preseason_futures_usd_per_bushel']
        - row1['harvest_futures_usd_per_bushel']
    )
)
manual_july_pnl = (
    row1['fixed50_july_adjustment_contracts']
    * CONTRACT_SIZE_BUSHELS
    * (
        row1['july_futures_usd_per_bushel']
        - row1['harvest_futures_usd_per_bushel']
    )
)

print(f'Initial futures P&L = ${manual_initial_pnl:,.2f}')
print(f'July adjustment P&L = ${manual_july_pnl:,.2f}')
print(f'Total futures P&L = ${manual_initial_pnl + manual_july_pnl:,.2f}')

assert np.isclose(manual_initial_pnl, row1['fixed50_initial_futures_pnl_usd'])
assert np.isclose(manual_july_pnl, row1['fixed50_july_adjustment_pnl_usd'])
print('Scenario 1 futures P&L手算验证通过。')

## 9. 计算Transaction Cost

每一次买入或卖出一份合约算一个`contract-side`：

- 3月卖出21份：21 sides；
- 7月调整0份：0 sides；
- 收获期买回21份：21 sides。

$$TransactionCost=(|N_0|+|\Delta N|+|N_H|)\times\$25$$

In [ ]:
transaction_sides = (
    np.abs(initial_contracts)
    + np.abs(july_adjustment_contracts)
    + np.abs(final_contracts)
)
transaction_cost = transaction_sides * TRANSACTION_COST_PER_CONTRACT_SIDE

scenarios['fixed50_transaction_sides'] = transaction_sides
scenarios['fixed50_transaction_cost_usd'] = transaction_cost

print('Transaction sides:', np.unique(transaction_sides))
print('Transaction cost:', np.unique(transaction_cost))

## 10. 计算Margin Financing Cost

Initial margin不是手续费，它通常在平仓后退回；但被占用的资金有融资或机会成本。

本项目使用：

$$MarginCost=2{,}500\times6\%\times\left(|N_0|\frac{136}{365}+|N_H|\frac{108}{365}\right)$$

这是简化成本，不是某个broker的实际报价。

In [ ]:
margin_financing_cost = (
    INITIAL_MARGIN_PER_CONTRACT
    * MARGIN_FINANCING_RATE_ANNUAL
    * (
        np.abs(initial_contracts) * DAYS_PRESEASON_TO_JULY / 365.0
        + np.abs(final_contracts) * DAYS_JULY_TO_HARVEST / 365.0
    )
)

scenarios['fixed50_margin_financing_cost_usd'] = margin_financing_cost

print(f'Margin financing cost = ${margin_financing_cost[0]:,.2f} per 1,000-acre farm')
print(f'Margin financing cost = ${margin_financing_cost[0] / FARM_ACRES:.4f}/acre')

## 11. 计算Fixed 50%最终利润

$$GrossRevenueAfterHedge_i=CashRevenue_i+TotalFuturesPnL_i$$

$$\quad-TransactionCost_i-MarginFinancingCost_i$$

$$Fixed50Profit_i=GrossRevenueAfterHedge_i-ProductionCost_i$$

In [ ]:
gross_revenue_after_hedge = (
    scenarios['cash_revenue_usd'].to_numpy()
    + total_futures_pnl
    - transaction_cost
    - margin_financing_cost
)

fixed50_profit = (
    gross_revenue_after_hedge
    - scenarios['production_cost_usd'].to_numpy()
)
fixed50_profit_per_acre = fixed50_profit / FARM_ACRES

scenarios['fixed50_gross_revenue_after_hedge_usd'] = gross_revenue_after_hedge
scenarios['fixed50_profit_usd'] = fixed50_profit
scenarios['fixed50_profit_usd_per_acre'] = fixed50_profit_per_acre

print(scenarios[[
    'scenario_id',
    'cash_revenue_usd',
    'fixed50_total_futures_pnl_usd',
    'fixed50_transaction_cost_usd',
    'fixed50_margin_financing_cost_usd',
    'production_cost_usd',
    'fixed50_profit_usd',
    'fixed50_profit_usd_per_acre',
]].head(10).round(2).to_string(index=False))

## 12. Scenario 1完整利润桥接

In [ ]:
row1 = scenarios.iloc[0]
manual_fixed50_profit = (
    row1['cash_revenue_usd']
    + row1['fixed50_initial_futures_pnl_usd']
    + row1['fixed50_july_adjustment_pnl_usd']
    - row1['fixed50_transaction_cost_usd']
    - row1['fixed50_margin_financing_cost_usd']
    - row1['production_cost_usd']
)

print(f"Cash revenue:                 ${row1['cash_revenue_usd']:>12,.2f}")
print(f"+ Initial futures P&L:         ${row1['fixed50_initial_futures_pnl_usd']:>12,.2f}")
print(f"+ July adjustment P&L:        ${row1['fixed50_july_adjustment_pnl_usd']:>12,.2f}")
print(f"- Transaction cost:           ${row1['fixed50_transaction_cost_usd']:>12,.2f}")
print(f"- Margin financing cost:      ${row1['fixed50_margin_financing_cost_usd']:>12,.2f}")
print(f"- Production cost:            ${row1['production_cost_usd']:>12,.2f}")
print(f'Fixed 50% profit:             ${manual_fixed50_profit:>12,.2f}')

assert np.isclose(manual_fixed50_profit, row1['fixed50_profit_usd'])
print('Scenario 1完整利润验证通过。')

## 13. Over-hedging与Margin-call Proxy

### Over-hedging

如果最终套保bushels大于实际产量，农民可能变成净投机者：

$$Overhedged_i=HedgedBushels_i>ActualProduction_i$$

### Margin-call proxy

本项目只检查July和Harvest两个观察点。当期货空头的负MTM超过$50,000 liquidity reserve时标记为1。它不模拟每日价格路径或broker maintenance margin。

In [ ]:
overhedged = (
    scenarios['fixed50_final_hedged_bushels'].to_numpy()
    > scenarios['actual_production_bushels'].to_numpy()
)

july_mtm = (
    initial_contracts
    * CONTRACT_SIZE_BUSHELS
    * (f0 - f_july)
)
harvest_segment_mtm = (
    final_contracts
    * CONTRACT_SIZE_BUSHELS
    * (f_july - f_harvest)
)

july_margin_call = (-july_mtm) > MARGIN_LIQUIDITY_RESERVE
harvest_margin_call = (-harvest_segment_mtm) > MARGIN_LIQUIDITY_RESERVE
margin_call_proxy = july_margin_call | harvest_margin_call

scenarios['fixed50_overhedged'] = overhedged
scenarios['fixed50_margin_call_proxy'] = margin_call_proxy

print(f'Probability overhedged = {overhedged.mean():.2%}')
print(f'Probability margin-call proxy = {margin_call_proxy.mean():.2%}')

## 14. 必须通过的模型检查

In [ ]:
assert (initial_contracts == 21).all()
assert (july_adjustment_contracts == 0).all()
assert (final_contracts == 21).all()
assert np.allclose(july_adjustment_pnl, 0.0)
assert np.allclose(total_futures_pnl, initial_futures_pnl)
assert np.allclose(transaction_cost, 1_050.0)
assert np.allclose(margin_financing_cost, 2_105.753424657534)
assert np.allclose(
    initial_futures_pnl,
    initial_contracts * CONTRACT_SIZE_BUSHELS * (f0 - f_harvest),
)
assert np.allclose(
    fixed50_profit,
    scenarios['unhedged_profit_usd'].to_numpy()
    + total_futures_pnl
    - transaction_cost
    - margin_financing_cost,
)

print('全部模型检查通过。')

## 15. 比较Unhedged与Fixed 50%风险指标

In [ ]:
def risk_summary(series):
    series = pd.Series(series)
    p5 = series.quantile(LOWER_TAIL_PROBABILITY)
    lower_tail = series[series <= p5]
    return pd.Series({
        'Mean': series.mean(),
        'Std Dev': series.std(ddof=1),
        'Probability Below Zero': (series < 0).mean(),
        'P5': p5,
        'CVaR 5%': lower_tail.mean(),
        'Median': series.median(),
        'P95': series.quantile(0.95),
        'Minimum': series.min(),
        'Maximum': series.max(),
    })

comparison = pd.DataFrame({
    'Unhedged': risk_summary(scenarios['unhedged_profit_usd_per_acre']),
    'Fixed 50%': risk_summary(fixed50_profit_per_acre),
}).T

print(comparison.round(4).to_string())

## 16. 把变化翻译成一句人话

我们分别查看平均利润、波动、亏损概率、P5和CVaR的变化。

In [ ]:
unhedged_profit_per_acre = scenarios['unhedged_profit_usd_per_acre']
u_p5 = unhedged_profit_per_acre.quantile(0.05)
f_p5 = pd.Series(fixed50_profit_per_acre).quantile(0.05)
u_cvar = unhedged_profit_per_acre[unhedged_profit_per_acre <= u_p5].mean()
f_cvar = pd.Series(fixed50_profit_per_acre)[fixed50_profit_per_acre <= f_p5].mean()

mean_change = fixed50_profit_per_acre.mean() - unhedged_profit_per_acre.mean()
sd_reduction = 1 - fixed50_profit_per_acre.std(ddof=1) / unhedged_profit_per_acre.std(ddof=1)
loss_probability_change_pp = 100 * (
    (fixed50_profit_per_acre < 0).mean()
    - (unhedged_profit_per_acre < 0).mean()
)

print(f'Expected profit change = ${mean_change:.2f}/acre')
print(f'Profit standard-deviation reduction = {sd_reduction:.2%}')
print(f'Loss-probability change = {loss_probability_change_pp:.2f} percentage points')
print(f'P5 improvement = ${f_p5 - u_p5:.2f}/acre')
print(f'CVaR 5% improvement = ${f_cvar - u_cvar:.2f}/acre')

## 17. 画出两种利润分布

如果Fixed 50%的分布更窄，说明它降低了利润波动。

In [ ]:
plt.figure(figsize=(10, 5))
plt.hist(
    unhedged_profit_per_acre,
    bins=45,
    alpha=0.55,
    label='Unhedged',
    color='#A5A5A5',
)
plt.hist(
    fixed50_profit_per_acre,
    bins=45,
    alpha=0.60,
    label='Fixed 50%',
    color='#4472C4',
)
plt.axvline(0, color='#C00000', linestyle='--', label='Break-even')
plt.xlabel('Profit (USD per acre)')
plt.ylabel('Number of simulations')
plt.title('Unhedged vs Fixed 50% Profit Distribution')
plt.legend()
plt.tight_layout()
plt.show()

## 18. 可重复性检查

In [ ]:
fixed50_series = pd.Series(fixed50_profit_per_acre)
fixed50_p5 = fixed50_series.quantile(0.05)
fixed50_cvar5 = fixed50_series[fixed50_series <= fixed50_p5].mean()

expected = {
    'mean_profit_per_acre': 11.913344135141102,
    'std_profit_per_acre': 80.05894846373614,
    'probability_below_zero': 0.4311,
    'p5_profit_per_acre': -144.97667145038278,
    'cvar5_profit_per_acre': -154.8015607295035,
    'mean_initial_futures_pnl_farm': 16663.558160431203,
    'probability_margin_call_proxy': 0.3716,
}

actual = {
    'mean_profit_per_acre': fixed50_series.mean(),
    'std_profit_per_acre': fixed50_series.std(ddof=1),
    'probability_below_zero': (fixed50_series < 0).mean(),
    'p5_profit_per_acre': fixed50_p5,
    'cvar5_profit_per_acre': fixed50_cvar5,
    'mean_initial_futures_pnl_farm': initial_futures_pnl.mean(),
    'probability_margin_call_proxy': margin_call_proxy.mean(),
}

for key in expected:
    assert np.isclose(actual[key], expected[key], atol=1e-10), (key, actual[key], expected[key])

print('可重复性检查通过。')
print(pd.DataFrame({'Expected': expected, 'Actual': actual}).round(6).to_string())

## 19. 保存第8课结果

In [ ]:
OUTPUT_DIR = Path.cwd() / 'lesson_08_outputs'
OUTPUT_DIR.mkdir(exist_ok=True)

scenario_path = OUTPUT_DIR / 'fixed_50pct_hedge_10000.csv'
summary_path = OUTPUT_DIR / 'step_08_fixed50_summary.json'

scenarios.to_csv(scenario_path, index=False)

summary_for_json = {
    'strategy': 'Fixed 50%',
    'n_simulations': N_SIMULATIONS,
    'initial_contracts': int(initial_contracts[0]),
    'july_adjustment_contracts': 0,
    'final_contracts': int(final_contracts[0]),
    'contract_size_bushels': CONTRACT_SIZE_BUSHELS,
    'transaction_cost_usd_per_contract_side': TRANSACTION_COST_PER_CONTRACT_SIDE,
    'margin_financing_cost_usd_1000_acre_farm': float(margin_financing_cost[0]),
    'profit_summary_usd_per_acre': {
        k: float(v) for k, v in risk_summary(fixed50_profit_per_acre).items()
    },
    'probability_overhedged': float(overhedged.mean()),
    'probability_margin_call_proxy': float(margin_call_proxy.mean()),
}

summary_path.write_text(json.dumps(summary_for_json, indent=2), encoding='utf-8')

print('已保存:', scenario_path)
print('已保存:', summary_path)

## 20. 本课结论与限制

Fixed 50%把profit standard deviation从约$153.73/acre降至$80.06/acre，并显著改善P5和CVaR 5%，说明它在当前情景下提供了明显的downside protection。

但要谨慎解释平均利润上升：本次模拟的Harvest futures平均低于3月价格，因此short futures产生正的平均P&L。这是校准样本和价格模型的结果，不代表套保天然创造正期望收益。

Margin-call proxy约37.16%，提醒我们：即使最终套保降低利润风险，价格上涨途中仍可能需要追加现金。该proxy只观察July和Harvest节点，不是完整的每日保证金模型。

现在还不能宣布Fixed 50%是最佳策略，因为尚未同时比较25%、75%、100%以及adaptive策略。

**下一课：把Fixed 0/25/50/75/100%全部放进同一框架，比较expected profit、standard deviation、P5、CVaR、over-hedging和margin liquidity risk。**